In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

In [3]:
# Function to convert Excel serial date to datetime (if needed)
def excel_to_datetime(serial):
    try:
        serial = int(serial)
        return datetime(1899, 12, 30) + timedelta(days=serial)
    except:
        return pd.NaT

In [4]:
# Step 2: Load and Inspect KSE Data (unchanged)
file_path = 'kse-30-basic.xlsx'
df_org = pd.read_excel(file_path)
df = df_org.copy()
df.columns = df.columns.str.lower().str.strip().str.replace(' ', '_')
if not pd.api.types.is_datetime64_any_dtype(df['date']):
    df['date'] = df['date'].apply(excel_to_datetime)
df = df.dropna(subset=['date', 'company', 'price', 'idx_wt_%', 'volume'])
df = df.sort_values(['date', 'company']).reset_index(drop=True)
df_cleaned = df.copy()

In [5]:
df_cleaned.head()

,date,symbol,company,price,idx_wt_%,volume
0,2020-01-01,BAHL,Bank Al-Habib Ltd.,77.33,4.29,257500
1,2020-01-01,BAFL,Bank Alfalah Ltd.,46.74,2.55,1128000
2,2020-01-01,BOP,Bank Of Punjab.,11.73,1.01,7507000
3,2020-01-01,DGKC,D. G. Khan Cement Co. Ltd.,75.23,1.27,1501000
4,2020-01-01,ENGRO,Engro Corporation Limited.,354.28,8.63,231900


In [6]:
# Compute daily aggregates
df_cleaned['weighted_price'] = df_cleaned['price'] * df_cleaned['idx_wt_%']
daily_agg = df_cleaned.groupby('date').agg(
    total_volume=('volume', 'sum'),
    weighted_return=('weighted_price', 'sum')
).reset_index()

In [7]:
df_cleaned.head()

,date,symbol,company,price,idx_wt_%,volume,weighted_price
0,2020-01-01,BAHL,Bank Al-Habib Ltd.,77.33,4.29,257500,331.7457
1,2020-01-01,BAFL,Bank Alfalah Ltd.,46.74,2.55,1128000,119.1870
2,2020-01-01,BOP,Bank Of Punjab.,11.73,1.01,7507000,11.8473
3,2020-01-01,DGKC,D. G. Khan Cement Co. Ltd.,75.23,1.27,1501000,95.5421
4,2020-01-01,ENGRO,Engro Corporation Limited.,354.28,8.63,231900,3057.4364


In [8]:
daily_agg.head()

,date,total_volume,weighted_return
0,2020-01-01,55169900,21114.9944
1,2020-01-02,151334700,21553.9228
2,2020-01-03,130968430,21457.5729
3,2020-01-06,96604890,20966.0359
4,2020-01-07,70796630,21329.3670


In [9]:
# Resample to monthly
daily_agg['date'] = pd.to_datetime(daily_agg['date'])
daily_agg.set_index('date', inplace=True)
monthly_df = daily_agg.resample('ME').agg({
    'total_volume': 'sum',
    'weighted_return': 'mean'
}).reset_index()

monthly_df['log_return'] = np.log(monthly_df['weighted_return'] / monthly_df['weighted_return'].shift(1))

In [10]:
monthly_df.head()

,date,total_volume,weighted_return,log_return
0,2020-01-31,2173067550,21788.892100,NaN
1,2020-02-29,1224806920,20260.168042,-0.072744
2,2020-03-31,2110137290,17200.543614,-0.163716
3,2020-04-30,2027859811,17004.016473,-0.011491
4,2020-05-31,1143466534,18689.350125,0.094504


In [11]:
# Load and process funds_data for flows
fund_sheets = ['AKD', 'NBP', 'NTI']
fund_flows = []
for sheet in fund_sheets:
    fund_df = pd.read_excel('funds_data.xlsx', sheet_name=sheet)
    fund_df.columns = fund_df.columns.str.lower().str.strip().str.replace(' ', '_')
    if not pd.api.types.is_datetime64_any_dtype(fund_df['date']):
        fund_df['date'] = fund_df['date'].apply(excel_to_datetime)
    fund_df = fund_df.dropna(subset=['date', 'nav', 'aum'])
    fund_df = fund_df.sort_values('date').reset_index(drop=True)
    fund_df['nav_ratio'] = fund_df['nav'] / fund_df['nav'].shift(1)
    fund_df['flow'] = fund_df['aum'] - (fund_df['aum'].shift(1) * fund_df['nav_ratio'])
    fund_df['flow'] = fund_df['flow'].fillna(0)
    fund_flows.append(fund_df[['date', 'flow']])

all_fund_flows = pd.concat(fund_flows)
all_fund_flows = all_fund_flows.groupby('date')['flow'].sum().reset_index()
all_fund_flows['date'] = pd.to_datetime(all_fund_flows['date'])
all_fund_flows.set_index('date', inplace=True)
total_monthly_flow = all_fund_flows.resample('ME')['flow'].sum().reset_index(name='total_fund_flow')

In [12]:
all_fund_flows.head()

,flow
date,
2020-03-18,0.000000
2020-03-24,0.239999
2020-03-25,0.489966
2020-03-26,0.340733
2020-03-27,-0.034697


In [13]:
total_monthly_flow.head()

,date,total_fund_flow
0,2020-03-31,0.876717
1,2020-04-30,35.484107
2,2020-05-31,-1.618847
3,2020-06-30,3.255424
4,2020-07-31,-0.899210


In [14]:
# Load and process macro_data
oil_df = pd.read_excel('macro_data.xlsx', sheet_name='OIL')
oil_df.columns = oil_df.columns.str.lower().str.strip().str.replace(' ', '_')
if not pd.api.types.is_datetime64_any_dtype(oil_df['date']):
    oil_df['date'] = oil_df['date'].apply(excel_to_datetime)
oil_df = oil_df.dropna(subset=['date'])
oil_df = oil_df.groupby('date')['price'].mean().reset_index()
oil_df['date'] = pd.to_datetime(oil_df['date'])
oil_df.set_index('date', inplace=True)
monthly_oil = oil_df.resample('ME')['price'].mean().reset_index(name='oil_price')

In [15]:
oil_df.head()

,price
date,
2021-01-04,51.09
2021-01-05,53.60
2021-01-06,54.30
2021-01-07,54.38
2021-01-08,55.99


In [16]:
monthly_oil.head()

,date,oil_price
0,2021-01-31,55.321500
1,2021-02-28,62.281500
2,2021-03-31,65.702174
3,2021-04-30,65.328571
4,2021-05-31,68.309048


In [17]:
ir_df = pd.read_excel('macro_data.xlsx', sheet_name='IR')
ir_df.columns = ir_df.columns.str.lower().str.strip().str.replace(' ', '_')
if not pd.api.types.is_datetime64_any_dtype(ir_df['date']):
    ir_df['date'] = ir_df['date'].apply(excel_to_datetime)
ir_df = ir_df.dropna(subset=['date'])
ir_df = ir_df.groupby('date')['rate'].last().reset_index()
ir_df['date'] = pd.to_datetime(ir_df['date'])
ir_df.set_index('date', inplace=True)
ir_df.sort_index(inplace=True)
daily_index = pd.date_range(start=ir_df.index.min(), end=ir_df.index.max(), freq='D')
ir_df = ir_df.reindex(daily_index).ffill()
monthly_ir = ir_df.resample('ME')['rate'].last().reset_index()
if 'index' in monthly_ir.columns:
    monthly_ir.rename(columns={'index': 'date'}, inplace=True)
monthly_ir.rename(columns={'rate': 'interest_rate'}, inplace=True)

In [18]:
ir_df.head()

,rate
2021-09-21,7.25
2021-09-22,7.25
2021-09-23,7.25
2021-09-24,7.25
2021-09-25,7.25


In [21]:
monthly_ir.head()

,date,interest_rate
0,2021-09-30,7.25
1,2021-10-31,7.25
2,2021-11-30,8.75
3,2021-12-31,9.75
4,2022-01-31,9.75


In [22]:
usd_df = pd.read_excel('macro_data.xlsx', sheet_name='USD')
usd_df.columns = usd_df.columns.str.lower().str.strip().str.replace(' ', '_')
if not pd.api.types.is_datetime64_any_dtype(usd_df['date']):
    usd_df['date'] = usd_df['date'].apply(excel_to_datetime)
usd_df = usd_df.dropna(subset=['date'])
usd_df = usd_df.groupby('date')['usd'].mean().reset_index()
usd_df['date'] = pd.to_datetime(usd_df['date'])
usd_df.set_index('date', inplace=True)
monthly_usd = usd_df.resample('ME')['usd'].mean().reset_index(name='usd_exchange')

In [23]:
usd_df.head()

,usd
date,
2021-03-01,159.98
2021-03-02,160.33
2021-03-03,160.29
2021-03-04,160.01
2021-03-07,160.17


In [24]:
monthly_usd.head()

,date,usd_exchange
0,2021-03-31,160.369091
1,2021-04-30,159.028421
2,2021-05-31,156.388235
3,2021-06-30,153.340000
4,2021-07-31,153.535556


In [ ]:
# Merge to enhanced monthly df
enhanced_monthly = monthly_df.merge(total_monthly_flow, on='date', how='left')
enhanced_monthly = enhanced_monthly.merge(monthly_oil, on='date', how='left')
enhanced_monthly = enhanced_monthly.merge(monthly_ir, on='date', how='left')
enhanced_monthly = enhanced_monthly.merge(monthly_usd, on='date', how='left')

# Fund flow: months before fund data starts have no flow — treat as 0
enhanced_monthly['total_fund_flow'] = enhanced_monthly['total_fund_flow'].fillna(0)

# Macro columns (oil, interest rate, USD) start from 2021.
# bfill fills pre-2021 rows with the earliest real value we have;
# ffill then closes any remaining gaps within the series.
enhanced_monthly[['oil_price', 'interest_rate', 'usd_exchange']] = (
    enhanced_monthly[['oil_price', 'interest_rate', 'usd_exchange']]
    .bfill()
    .ffill()
)

# log_return for the first row is NaN (no prior month) — fill with 0
enhanced_monthly['log_return'] = enhanced_monthly['log_return'].fillna(0)

In [31]:
enhanced_monthly.head(50)

,date,total_volume,weighted_return,log_return,total_fund_flow,oil_price,interest_rate,usd_exchange
0,2020-01-31,2173067550,21788.892100,0.007563,-26.571311,80.561540,15.81,238.804379
1,2020-02-29,1224806920,20260.168042,-0.072744,-26.571311,80.561540,15.81,238.804379
2,2020-03-31,2110137290,17200.543614,-0.163716,0.876717,80.561540,15.81,238.804379
3,2020-04-30,2027859811,17004.016473,-0.011491,35.484107,80.561540,15.81,238.804379
4,2020-05-31,1143466534,18689.350125,0.094504,-1.618847,80.561540,15.81,238.804379
5,2020-06-30,1310837237,18915.900682,0.012049,3.255424,80.561540,15.81,238.804379
6,2020-07-31,3143791324,20393.112395,0.075194,-0.899210,80.561540,15.81,238.804379
7,2020-08-31,2999557437,22065.068795,0.078799,-2.206613,80.561540,15.81,238.804379
8,2020-09-30,3410834203,22145.499409,0.003639,-0.880381,80.561540,15.81,238.804379
9,2020-10-31,3772329907,20735.210524,-0.065801,-2.452764,80.561540,15.81,238.804379


In [ ]:
# Winsorize total_fund_flow at ±3 standard deviations
# The April 2022 outlier (-62.7) is a one-off political event, not a learnable pattern
mean_flow = enhanced_monthly['total_fund_flow'].mean()
std_flow = enhanced_monthly['total_fund_flow'].std()
lower_bound = mean_flow - 3 * std_flow
upper_bound = mean_flow + 3 * std_flow
enhanced_monthly['total_fund_flow'] = enhanced_monthly['total_fund_flow'].clip(lower=lower_bound, upper=upper_bound)
print(f"Winsorize bounds: [{lower_bound:.2f}, {upper_bound:.2f}]")
print(f"Flow range after winsorize: [{enhanced_monthly['total_fund_flow'].min():.2f}, {enhanced_monthly['total_fund_flow'].max():.2f}]")

In [32]:
# Lagged features
enhanced_monthly['lag_volume'] = enhanced_monthly['total_volume'].shift(1)
enhanced_monthly['lag_return'] = enhanced_monthly['log_return'].shift(1)
enhanced_monthly['lag_oil'] = enhanced_monthly['oil_price'].shift(1)
enhanced_monthly['lag_ir'] = enhanced_monthly['interest_rate'].shift(1)
enhanced_monthly['lag_usd'] = enhanced_monthly['usd_exchange'].shift(1)

enhanced_monthly = enhanced_monthly.dropna()

In [33]:
enhanced_monthly.head()

,date,total_volume,weighted_return,log_return,total_fund_flow,oil_price,interest_rate,usd_exchange,lag_volume,lag_return,lag_oil,lag_ir,lag_usd
1,2020-02-29,1224806920,20260.168042,-0.072744,-26.571311,80.56154,15.81,238.804379,2.173068e+09,0.007563,80.56154,15.81,238.804379
2,2020-03-31,2110137290,17200.543614,-0.163716,0.876717,80.56154,15.81,238.804379,1.224807e+09,-0.072744,80.56154,15.81,238.804379
3,2020-04-30,2027859811,17004.016473,-0.011491,35.484107,80.56154,15.81,238.804379,2.110137e+09,-0.163716,80.56154,15.81,238.804379
4,2020-05-31,1143466534,18689.350125,0.094504,-1.618847,80.56154,15.81,238.804379,2.027860e+09,-0.011491,80.56154,15.81,238.804379
5,2020-06-30,1310837237,18915.900682,0.012049,3.255424,80.56154,15.81,238.804379,1.143467e+09,0.094504,80.56154,15.81,238.804379


In [ ]:
from sklearn.model_selection import TimeSeriesSplit

features = ['lag_volume', 'lag_return', 'lag_oil', 'lag_ir', 'lag_usd']
X = enhanced_monthly[features]
y = enhanced_monthly['total_fund_flow']

tscv = TimeSeriesSplit(n_splits=5)

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

rf = RandomForestRegressor(n_estimators=100, random_state=42)

fold_rmses = []
for fold, (train_idx, test_idx) in enumerate(tscv.split(X), start=1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    rf.fit(X_train, y_train)
    preds = rf.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    fold_rmses.append(rmse)
    print(f"Fold {fold} | test rows: {len(y_test)} | RMSE: {rmse:.4f}")

print(f"\nMean RMSE across folds: {np.mean(fold_rmses):.4f}")
print(f"Std  RMSE across folds: {np.std(fold_rmses):.4f}")

# Refit on full data for next-month prediction
rf.fit(X, y)
last_row = enhanced_monthly.iloc[-1][features].values.reshape(1, -1)
next_pred = rf.predict(last_row)
print(f"\nPredicted next fund flow: {next_pred[0]:.4f}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(5, 1, figsize=(12, 18), sharex=False)
fig.suptitle('Fund Flow Predictions — TimeSeriesSplit (5 Folds)', fontsize=14, y=1.01)

for fold, (train_idx, test_idx) in enumerate(tscv.split(X), start=1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    rf_plot = RandomForestRegressor(n_estimators=100, random_state=42)
    rf_plot.fit(X_train, y_train)
    preds = rf_plot.predict(X_test)

    ax = axes[fold - 1]
    ax.plot(enhanced_monthly['date'].iloc[test_idx], y_test.values, marker='o', label='Actual', color='steelblue')
    ax.plot(enhanced_monthly['date'].iloc[test_idx], preds, marker='x', linestyle='--', label='Predicted', color='tomato')
    ax.set_title(f'Fold {fold} | RMSE: {fold_rmses[fold-1]:.4f}')
    ax.set_ylabel('Fund Flow')
    ax.legend(fontsize=8)
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Train/test split
split = int(len(enhanced_monthly) * 0.8)
train = enhanced_monthly.iloc[:split]
test = enhanced_monthly.iloc[split:]

features = ['lag_volume', 'lag_return', 'lag_oil', 'lag_ir', 'lag_usd']
X_train, y_train = train[features], train['total_fund_flow']
X_test, y_test = test[features], test['total_fund_flow']

# Model (Random Forest; swap for XGBoost if preferred)
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
preds_rf = rf.predict(X_test)

# Evaluate
mse = mean_squared_error(y_test, preds_rf)
rmse = np.sqrt(mse)
print(f"RMSE on test: {rmse}")

# Predict next
last_row = enhanced_monthly.iloc[-1][features].values.reshape(1, -1)
next_pred = rf.predict(last_row)
print(f"Predicted next fund flow: {next_pred[0]}")

# Plot predictions (optional)
plt.figure(figsize=(10, 5))
plt.plot(test['date'], y_test, label='Actual Flows')
plt.plot(test['date'], preds_rf, label='Predicted (RF)')
plt.title('Fund Flow Prediction (Test Set)')
plt.legend()
plt.show()

from sklearn.metrics import r2_score

r2 = r2_score(y_test, preds_rf)
print(f"R-squared on test: {r2:.4f}")

n = len(y_test)          # number of observations
p = X_test.shape[1]      # number of predictors
adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)
print(f"Adjusted R-squared on test: {adj_r2:.4f}")

In [38]:
# Suggestion
threshold = enhanced_monthly['total_fund_flow'].mean()
if next_pred[0] > threshold:
    print("Predicted inflow → Suggest overweight high-momentum/volume stocks")
    recent_date = df_cleaned['date'].max()
    recent = df_cleaned[df_cleaned['date'] == recent_date].sort_values('volume', ascending=False).head(10)
    print(recent[['company', 'volume', 'idx_wt_%']])
else:
    print("Predicted outflow/flat → Suggest defensive or equal-weight")

Predicted inflow → Suggest overweight high-momentum/volume stocks
                                   company     volume  idx_wt_%
42302                     Bank Of Punjab.   191997025  1.148725
42319                   Pak Elektron Ltd.   132355135  0.840521
42311          Hub Power Company Limited.    13042153  5.949837
42317          National Bank Of Pakistan.    12492878  2.904485
42306                Fauji Cement Co Ltd.     8736512  1.457303
42322              Pakistan Refinery Ltd.     8208613  0.227474
42303          D. G. Khan Cement Co. Ltd.     7089151  1.588068
42323          Pakistan State Oil Co Ltd.     6837555  3.078313
42321         Pakistan Petroleum Limited.     6827405  3.821311
42318  Oil & Gas Development Company Ltd.     6107600  4.979638
